# 경구약제 검출 — 정정판 파이프라인 (run_experiment_fixed)

기존 문제 진단:
- **검증셋 누수**: 같은 약 조합을 각도만 바꿔 찍은 사진이 train/val에 동시에 들어가
  val mAP가 0.9로 부풀려졌음(실제 일반화 성능 아님).
- **도메인/카테고리 불일치**: AIHub 임의 category_id로 학습 → 제출은 Kaggle K-코드 필요.

정정 내용:
1. **Kaggle 제공 train 데이터로 직접 학습**(테스트와 동일 분포, 올바른 category_id 자동 생성).
2. **조합(combo) 단위 그룹 분할**로 train/val 누수 0.
3. 전체 데이터 + 충분한 epoch.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
from src.joelchoi.data.kaggle_converter import convert_kaggle_to_coco
from src.joelchoi.data.subset import create_subset, combo_group_key
from src.joelchoi.data.coco_to_yolo import prepare_yolo_dataset
from src.joelchoi.train_yolo import run_yolo_experiment
from src.joelchoi.utils import load_config

CONFIGS_DIR = PROJECT_ROOT / "configs" / "joelchoi"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments" / "joelchoi"
DATA_DIR = PROJECT_ROOT / "data" / "joelchoi"
YOLO_DIR = DATA_DIR / "yolo_kaggle_full"  # 정정판 데이터셋 출력 위치
print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/joelchoi/study/projects/project1_3team


## 1단계 — Kaggle train 데이터 → COCO
`category_id`는 K-코드 숫자(K-001900 → 1900)로 채점 기준과 동일. 조합2(테스트)는 train에 없음.

In [3]:
coco = convert_kaggle_to_coco()
print("클래스 수:", len(coco["categories"]))

Kaggle 학습 데이터 변환 완료: 이미지 232장, 어노테이션 763개, 클래스 56개
클래스 수: 56


## 2단계 — 누수 없는 그룹 분할 + YOLO 데이터셋 생성
`create_subset`이 조합 접두사 단위로 분할(누수 0 자동 검증). `prepare_yolo_dataset`이
`data.yaml`과 `class_map.json`(yolo_idx → category_id)을 함께 생성.

In [4]:
TIER = "full"  # 전체 데이터. 빠른 점검은 "medium"
train_coco, val_coco = create_subset(coco, tier=TIER, test_size=0.2, seed=42)

# 누수 재확인(조합 교집합 0이어야 함)
g = lambda c: {combo_group_key(i["file_name"]) for i in c["images"]}
print("train/val 공유 조합:", len(g(train_coco) & g(val_coco)))

yaml_path = prepare_yolo_dataset(train_coco, val_coco, YOLO_DIR, symlink=True)
CLASS_MAP = YOLO_DIR / "class_map.json"
print("data.yaml:", yaml_path)
print("class_map:", CLASS_MAP)

[full] train: 187장(91조합), val: 45장(23조합) | 조합 누수 0
train/val 공유 조합: 0
YOLO yaml 저장: /Users/joelchoi/study/projects/project1_3team/data/joelchoi/yolo_kaggle_full/data.yaml
클래스 매핑 저장: /Users/joelchoi/study/projects/project1_3team/data/joelchoi/yolo_kaggle_full/class_map.json
YOLO 데이터셋 준비 완료: train 187장, val 45장
data.yaml: /Users/joelchoi/study/projects/project1_3team/data/joelchoi/yolo_kaggle_full/data.yaml
class_map: /Users/joelchoi/study/projects/project1_3team/data/joelchoi/yolo_kaggle_full/class_map.json


## 3단계 — 학습 (YOLO11n)

In [5]:
cfg = load_config(CONFIGS_DIR / "experiment" / "exp010_yolo11n_kaggle.yaml")
cfg["data"]["subset"] = TIER
metrics = run_yolo_experiment(cfg, yaml_path, EXPERIMENTS_DIR)
print(metrics)  # 이제 val mAP는 '현실적' 수치 — 제출 점수와 비슷해야 정상

Ultralytics 8.4.83 🚀 Python-3.14.6 torch-2.12.1 CPU (Apple M2 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/joelchoi/study/projects/project1_3team/data/joelchoi/yolo_kaggle_full/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp010_yolo11n_kaggle, nbs=64, nms=False, opset=None, opti

## 4단계 — 테스트 추론 & 제출 파일 생성

In [6]:
from src.joelchoi.inference import run_inference, save_submission

EXP_NAME = cfg["experiment"]["name"]
KAGGLE_DATA = (
    Path.home()
    / ".cache/kagglehub/competitions/ai12-level1-project/sprint_ai_project1_data"
)
WEIGHTS = EXPERIMENTS_DIR / EXP_NAME / "weights" / "best.pt"
SUBMISSION = PROJECT_ROOT / "submissions" / "joelchoi" / f"{EXP_NAME}_submission.csv"

preds = run_inference(
    weights_path=WEIGHTS,
    class_map_path=CLASS_MAP,
    test_dir=KAGGLE_DATA / "test_images",
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=cfg["data"]["img_size"],
)
save_submission(preds, SUBMISSION)
print("제출 파일:", SUBMISSION)

추론 대상: 842장
총 예측 수: 3324개 (이미지당 평균 3.9개)
제출 파일 저장: /Users/joelchoi/study/projects/project1_3team/submissions/joelchoi/exp010_yolo11n_kaggle_submission.csv (3324개 예측)
제출 파일: /Users/joelchoi/study/projects/project1_3team/submissions/joelchoi/exp010_yolo11n_kaggle_submission.csv


## 5단계 — 제출 파일 점검

In [7]:
import pandas as pd

df = pd.read_csv(SUBMISSION)
test_ids = {int(p.stem) for p in (KAGGLE_DATA / "test_images").glob("*.png")}
pred_ids = set(df["image_id"].unique())
print(
    f"행 {len(df)}, 예측된 이미지 {len(pred_ids)}/{len(test_ids)}, 고유 category {df['category_id'].nunique()}"
)
print("검출 없는 이미지:", len(test_ids - pred_ids))
df.head()

행 3324, 예측된 이미지 842/842, 고유 category 51
검출 없는 이미지: 0


,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,1900,158,253,204,124,0.99
1,2,1,27926,592,666,265,489,0.97
2,3,1,30308,169,745,184,290,0.88
3,4,1,31885,553,66,391,414,0.52
4,5,3,1900,142,243,197,126,0.99
